# Part 1: Class Summary by Buffer

**Calculate total area and percent occupied by class for buffers of CWMP and GLFWRA bird count locations.**

_Buffers_: 200m, 500m, 1km, 2km  
_Data_: land cover and wetland type & code  
_Years_: 2010, 2015, 2020, 2023  
_Deliverable_: CSV file for each bird count dataset containing class summaries for each buffer for each location.

## Data

### Land Cover

##### [National Land Cover Database (NLCD)](https://www.usgs.gov/data/annual-national-land-cover-database-nlcd-collection-1-products-ver-11-june-2025)  
*source*: United States Geological Survey (USGS)  
*years*: 2010, 2015, 2020, 2023  
*metadata*: https://www.mrlc.gov/downloads/sciweb1/shared/mrlc/metadata/Annual_NLCD_LndCov_2024_CU_C1V1.xml
*how to cite*:  
U.S. Geological Survey (USGS), 2024, Annual NLCD Collection 1 Science Products (ver. 1.1, June 2025): U.S. Geological Survey data release, https://doi.org/10.5066/P94UXNTS.


##### [North American Land Change Monitoring System](https://www.cec.org/north-american-land-change-monitoring-system/)  
*source*: Commission for Environmental Cooperation (CEC)  
*years*: 2010, 2015, 2020  
*metadata*: [NA_landcover_2020v2_30m_metadata.html](metadata/NA_landcover_2020v2_30m_metadata.html)  
*how to cite*:  
Commission for Environmental Cooperation (CEC). 2024. "North American Environmental Atlas - Land Cover 2020 30m". North American Land Change Monitoring System. Canada Centre for Remote Sensing (CCRS), U.S. Geological Survey (USGS), Comisión Nacional para el Conocimiento y Uso de la Biodiversidad (CONABIO), Comisión Nacional Forestal (CONAFOR), Instituto Nacional de Estadística y Geografía (INEGI). Ed. 2.0, Raster digital data [30-m]. Available at https://www.cec.org/north-american-environmental-atlas/land-cover-30m-2020/

### Wetlands

##### [National Wetlands Inventory](https://www.fws.gov/program/national-wetlands-inventory)
*source*: United States Fish and Wildlife Service (USFWS)  
*years*: continually updated  
*metadata*: https://documentst.ecosphere.fws.gov/wetlands/data/metadata/FWS_Wetlands.xml  
*how to cite*:  
U. S. Fish and Wildlife Service. May 1st, 2024. National Wetlands Inventory website. U.S. Department of the Interior, Fish and Wildlife Service, Washington, D.C.

### Bird Count Locations

Bird count locations were provided by Annie & Isabel. From the original methods notes:
- Great Lakes Coastal Wetland Monitoring Program (CWMP) bird point count locations (PCL): [CWMP_PCL_all.xlsx](https://docs.google.com/spreadsheets/d/1-FZrtUYZS9HEFVKIO94fKBhGXi5KQDon)
- GLFWRA bird point count locations (PCL): GLFWRA_PCL_all.csv


In [ ]:
# Weird proj Issue
# 0) RESTART the Jupyter kernel first (not just "Run All").
# 1) Run this as the very first cell:
import os, sys

# Drop any OSGeo4W bleed-through
for k in ["PROJ_LIB", "PROJ_DATA", "GDAL_DATA"]:
    if os.environ.get(k, "").lower().find("osgeo4w") != -1:
        os.environ.pop(k, None)
os.environ["PATH"] = ";".join(
    p for p in os.environ["PATH"].split(";") if "OSGeo4W" not in p
)

from pyproj import CRS, datadir

crs4326 = CRS.from_epsg(4326)
print("CRS ok:", crs4326)
print("pyproj datadir:", datadir.get_data_dir())
print("PROJ_LIB:", os.environ.get("PROJ_LIB"))

In [3]:
# IMPORTS

# external
from pathlib import Path
import geopandas as gpd
import pandas as pd
from tqdm.auto import tqdm

# local
from libs.land_cover import (
    nlcd_sesh_setup,
    process_one_landcover,
    get_nlcd_paths,
)
from libs.nwi import process_one_nwi

In [4]:
# CONSTANTS

# buffer sizes
buffers = [200, 500, 1000, 2000]

# years
years = [2010, 2015, 2020, 2023]

# project CRS
crs = "EPSG:5070"

In [5]:
# Paths

# cwmp bird count locations
cwmp_path = "data/CWMP_PCL_all.xlsx - CWMP_bird_point_count_locations.csv"
# glfwra (betty) locations
glfwra_path = "data/GLFWRA_PCL_all.csv"

# nalcms land cover paths
# nalcms_paths = {
#     2010: r"data/nalcms/land_cover_2010v3_30m_tif/NA_NALCMS_landcover_2010v3_30m/data/NA_NALCMS_landcover_2010v3_30m.tif",
#     2015: r"data/nalcms/land_cover_2015v4_30m_tif/NA_NALCMS_landcover_2015v4_30m/data/NA_NALCMS_landcover_2015v4_30m.tif",
#     2020: r"data/nalcms/land_cover_2020v2_30m_tif/NA_NALCMS_landcover_2020v2_30m/data/NA_NALCMS_landcover_2020v2_30m.tif",
# }
nalcms_paths = {
    2010: "data/nalcms/land_cover_2010v3_30m_tif/NA_NALCMS_landcover_2010v3_30m/data/NA_NALCMS_landcover_2010v3_30m.tif",
    2015: "data/nalcms/land_cover_2015v4_30m_tif/NA_NALCMS_landcover_2015v4_30m/data/NA_NALCMS_landcover_2015v4_30m.tif",
    2020: "data/nalcms/land_cover_2020v2_30m_tif/NA_NALCMS_landcover_2020v2_30m/data/NA_NALCMS_landcover_2020v2_30m.tif",
}

# nwi geopackage path listing
nwi_path_json = "data/nwi_filelist.json"

In [ ]:
# set up nlcd sesh
nlcd_sesh = nlcd_sesh_setup()

# get nlcd paths
nlcd_paths = get_nlcd_paths()

## Load bird count locations

- [coastal wetlands](https://docs.google.com/spreadsheets/d/1-FZrtUYZS9HEFVKIO94fKBhGXi5KQDon)
  - field mapping
    - lat: latitude
    - lon: longitude


In [7]:
# load cwmp table
cwmp_df = pd.read_csv(cwmp_path)
cwmp_df.head()

,site_id,crew_name,year,date,sample,point_id,bearing,waypoint,longitude,latitude,coord_qual,comments
0,1,SUNY-B,2015,5/28/2015,1,1,140,AB0001.1.BE,-78.840043,43.312097,Excellent,NaN
1,1,SUNY-B,2020,6/1/2020,1,1,238,AB0001.1.AM,-78.840627,43.312285,Confirmed,NaN
2,4,SUNY-B,2015,5/28/2015,1,1,350,AB0004.1.BE,-78.744200,43.329043,Excellent,NaN
3,5,SUNY-B,2013,6/12/2013,1,1,183,B0005.PM,-78.715802,43.334753,Excellent,NaN
4,5,SUNY-B,2018,5/24/2018,1,1,180,AB0005.1.PM,-78.716118,43.335103,Excellent,NaN



- GLFWRA
  - field mapping
    - lat: Lat
    - lon: Lon

In [8]:
# load glfwra (betty) table
glfwra_df = pd.read_csv(glfwra_path)
display(glfwra_df.head())

,Source,Site,PID,Lat,Lon
0,"21_NRRI, CWM",21st Ave W,7049.1.2,46.757203,-92.125481
1,"21_NRRI, CWM",21st Ave W,7049.2.2,46.759090,-92.122097
2,"21_NRRI, CWM",21st Ave W,7049.1,46.763798,-92.123170
3,"21_NRRI, CWM",21st Ave W,7049.2,46.760550,-92.123525
4,CWM,40th Ave W,7048.1,46.739972,-92.153694


In [ ]:
bird_locs_field_mapping = {
    "CWMP": {
        "lat": "latitude",
        "lon": "longitude",
    },
    "GLFWRA": {
        "lat": "Lat",
        "lon": "Lon",
    },
}

## Process each location, buffer

1. create buffer
2. read in raster for buffer
3. calculate stats for buffer

- for each class:
  - count
  - area
  - percent of total

In [ ]:
# Main processing loop

# Process both sources explicitly to avoid df.equals() checks each row
sources = [("GLFWRA", glfwra_df), ("CWMP", cwmp_df)]

for source_name, df in sources:
    print(f"Processing {source_name}: {len(df)} bird count locations")

    # Resolve lat/lon column names and their tuple positions once
    lat_field = bird_locs_field_mapping[source_name]["lat"]
    lon_field = bird_locs_field_mapping[source_name]["lon"]
    lat_idx = 1 + df.columns.get_loc(
        lat_field
    )  # +1 because itertuples(index=True) prepends Index at [0]
    lon_idx = 1 + df.columns.get_loc(lon_field)

    rows_out = []  # one dict per input row

    # Fast row iteration
    for row in tqdm(
        df.itertuples(index=True, name=None),
        total=len(df),
        desc="Locations",
        dynamic_ncols=True,
    ):
        idx = row[0]
        lon = float(row[lon_idx])
        lat = float(row[lat_idx])

        # create point geometry
        point = gpd.points_from_xy([lon], [lat], crs="EPSG:4326")

        row_results = []
        for buffer_size in buffers:
            for year in years:
                # NLCD
                nlcd_result = process_one_landcover(
                    point=point,
                    buffer_size=buffer_size,
                    year=year,
                    source="nlcd",
                    nlcd_paths=nlcd_paths,
                    aws_sesh=nlcd_sesh,
                )
                if nlcd_result is not None:
                    row_results.append(nlcd_result)

                # NALCMS
                nalcms_result = process_one_landcover(
                    point=point,
                    buffer_size=buffer_size,
                    year=year,
                    source="nalcms",
                    nalcms_paths=nalcms_paths,
                )
                if nalcms_result is not None:
                    row_results.append(nalcms_result)

            # NWI (per buffer, not per year)
            nwi_result = process_one_nwi(
                point=point,
                buffer_size=buffer_size,
                source="nwi",
                crs=crs,
                nwi_path_json=nwi_path_json,
            )
            if nwi_result is not None:
                row_results.append(nwi_result)

        # Merge this location's dicts into one dict
        if row_results:
            merged = {k: v for d in row_results for k, v in d.items()}
            merged["_source_index"] = idx  # keep the original row index
            rows_out.append(merged)
        else:
            rows_out.append({"_source_index": idx})

    # Build a DataFrame aligned to original df
    new_cols = (
        pd.DataFrame(rows_out).set_index("_source_index").reindex(df.index)
    )
    results_df = df.join(new_cols)

    display(results_df.head())

    print(f"Completed processing for {source_name}. Exporting results...")
    output_dir = Path("data/output")
    output_dir.mkdir(parents=True, exist_ok=True)
    outpath = output_dir / f"{source_name.lower()}_class_summary_by_buffer.csv"
    results_df.to_csv(outpath, index=False)

## Output format

Output is in wide format, meaning the output spreadsheet has the same number of rows as the input spreadsheet. Output results have been added as new columns to the right of existing columns.

The format of the new columns is as:  
`<dataset>_buff<size>_yr<number>_<wetland code/>type>_<class>_<count/area/percent>`

### Some examples:

The percent of developed (open space) landcover according to the NLCD for the year 2010 within the 200m buffer would look like this:  
`nlcd_buff200_yr2010_developed__open_space_percent`

The area classified as urban according to the NALCMS for the year 2020 within the 200m buffer would look like this:  
`nalcms_buff200_yr2020_urban_area`

Wetlands with a class code of R2UBH occupy this percent of total area within the 500m buffer would look like this:  
`nwi_buff500_code_R2UBH_percent`
> *note the lack of year for NWI columns*

The percent of the 500m buffer with Riverine type wetlands would look like this:  
`nwi_buff500_type_riverine_percent`

### Column notes

- *count* is the number of pixels (for landcover) or wetland polygons
- *area* is in meters squared
- *percent* is a decimal (0-1)